# 01 · Baseline

How sycophantic is each model before any intervention? Runs the full harness on the **test** split
(capability guard, factual pressure, corrections/selectivity, multi-turn pressure, opinion sycophancy).

## Setup

Runtime → Change runtime type → **T4 GPU**. The repo is cloned to `/content/sycophancy` and results are written to
Google Drive so they survive a disconnect.

In [ ]:
import os
if not os.path.exists("/content/sycophancy"):
    !git clone https://github.com/namita-tj/sycophancy /content/sycophancy
%cd /content/sycophancy
!pip install -q -e ".[all]"

In [ ]:
# Gated models (Llama 3.2, Gemma 2) need a Hugging Face token with access accepted on the model page.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Keep results on Drive.
from google.colab import drive
drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/sycophancy/results"
os.makedirs(RESULTS, exist_ok=True)
!rm -rf results && ln -s $RESULTS results

In [ ]:
# Evaluation data (MMLU + Perez et al.) and synthetic training data. Only needed once per runtime.
if not os.path.exists("data/eval/pressure.jsonl"):
    !bash scripts/prepare_data.sh
!wc -l data/eval/*.jsonl data/train/*.jsonl

## Quick check on the simulated model (no GPU, a few seconds)

In [ ]:
!syco eval --model fake --data data/samples --split all --out results/fake-baseline

## Baseline for every model

In [ ]:
MODELS = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-1B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "google/gemma-2-2b-it",
    "allenai/OLMo-2-0425-1B-SFT",
    "allenai/OLMo-2-0425-1B-DPO",
    "allenai/OLMo-2-0425-1B-Instruct",
]
DTYPE = "fp16"
LIMIT = ""          # e.g. "--limit 100" for a first pass

In [ ]:
for m in MODELS:
    tag = m.split("/")[-1].lower()
    if os.path.exists(f"results/{tag}-baseline/scores.json"):
        print("skip", tag); continue
    !syco eval --model {m} --dtype {DTYPE} --data data/eval --split test {LIMIT} --out results/{tag}-baseline

In [ ]:
runs = " ".join(f"results/{m.split('/')[-1].lower()}-baseline" for m in MODELS)
!syco compare {runs} --out results/baselines.md
from IPython.display import Markdown
Markdown(open("results/baselines.md").read())

**What to look at:** flip rates under "Are you sure?" and wrong suggestions, selectivity on corrections
(0.5 means the model can't tell a right correction from a wrong one), the multi-turn survival curve, and whether
the OLMo SFT → DPO → Instruct checkpoints differ. Check the OLMo checkpoint names on the Hub before running.